In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Cache embeddings so you don't recompute for the same skill repeatedly
embedding_cache = {}

def get_embedding(skill):
    if skill not in embedding_cache:
        embedding_cache[skill] = embed_model.encode([skill])[0]
    return embedding_cache[skill]

def semantic_similarity(skill_a, skill_b):
    emb_a = get_embedding(skill_a)
    emb_b = get_embedding(skill_b)
    score = cosine_similarity([emb_a], [emb_b])[0][0]
    return float(score)

/opt/anaconda3/envs/geo-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 7406.24it/s]


In [9]:
skill_a=input("skill a:")
skill_b=input("skill b:")
print(semantic_similarity(skill_a,skill_b))

skill a: flask
skill b: python


0.47680604457855225


In [12]:
import re
import json
import numpy as np
import networkx as nx
import requests
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

# ─────────────────────────────────────────
# 1. SETUP
# ─────────────────────────────────────────

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_cache = {}
graph_cache = {}

client = Groq(api_key="API_KEY")

# ─────────────────────────────────────────
# 2. NORMALIZATION
# ─────────────────────────────────────────

def normalize_skill(skill):
    skill = skill.lower().strip()
    skill = re.sub(r'[^a-z0-9.#+\s]', '', skill).strip()
    return skill

# ─────────────────────────────────────────
# 3. EMBEDDINGS
# ─────────────────────────────────────────

def get_embedding(skill):
    if skill not in embedding_cache:
        embedding_cache[skill] = embed_model.encode([skill])[0]
    return embedding_cache[skill]

def semantic_similarity(skill_a, skill_b):
    emb_a = get_embedding(skill_a)
    emb_b = get_embedding(skill_b)
    score = cosine_similarity([emb_a], [emb_b])[0][0]
    return float(score)

# ─────────────────────────────────────────
# 4. GROQ LLM QUERY
# ─────────────────────────────────────────

def query_llm_for_implications(candidate_skill, required_skills):
    skill_list = ", ".join(required_skills)

    prompt = f"""
You are a technical skill expert.

Given that a candidate knows: "{candidate_skill}"
And the job requires these skills: [{skill_list}]

For each required skill, determine if knowing "{candidate_skill}" implies or demonstrates knowledge of that required skill.
Only include skills where there is a genuine technical implication (e.g., knowing Flask implies Python).
Do NOT include skills that are unrelated.

Respond ONLY with a valid JSON array. No explanation, no markdown, no extra text.
Format:
[
  {{"skill": "required_skill_name", "confidence": 0.95, "reason": "brief reason"}},
  ...
]

If "{candidate_skill}" implies nothing from the list, return an empty array: []
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",  # or "mixtral-8x7b-32768" or "gemma2-9b-it"
        messages=[{"role": "user", "content": prompt}],
        temperature=0,  # 0 for deterministic/consistent output
        max_tokens=1000
    )

    text = response.choices[0].message.content.strip()
    text = re.sub(r"```json|```", "", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print(f"  [WARN] Could not parse LLM response for '{candidate_skill}': {text}")
        return []

# ─────────────────────────────────────────
# 5. BUILD DYNAMIC GRAPH
# ─────────────────────────────────────────

def build_dynamic_graph(candidate_skills, required_skills):
    G = nx.DiGraph()
    all_required = [normalize_skill(s) for s in required_skills]

    for cand in candidate_skills:
        c = normalize_skill(cand)
        cache_key = (c, tuple(sorted(all_required)))

        if cache_key in graph_cache:
            implications = graph_cache[cache_key]
        else:
            print(f"  [LLM] Querying implications for: '{cand}'")
            implications = query_llm_for_implications(cand, required_skills)
            graph_cache[cache_key] = implications

        for item in implications:
            implied = normalize_skill(item["skill"])
            confidence = float(item["confidence"])
            reason = item.get("reason", "")
            if implied in all_required:
                G.add_edge(c, implied, weight=confidence, reason=reason)

    return G

# ─────────────────────────────────────────
# 6. SKILL SIMILARITY SCORE
# ─────────────────────────────────────────

def skill_similarity_score(candidate_skill, required_skill, graph, sem_threshold=0.72):
    c = normalize_skill(candidate_skill)
    r = normalize_skill(required_skill)

    # Case 1: Exact match
    if c == r:
        return 1.0, "exact match"

    # Case 2: Graph implication (LLM-generated)
    if graph.has_edge(c, r):
        weight = graph[c][r]['weight']
        reason = graph[c][r].get('reason', 'graph implication')
        return weight, reason

    # Case 3: Semantic similarity (synonym/abbreviation fallback)
    sem = semantic_similarity(c, r)
    if sem >= sem_threshold:
        return sem, f"semantically similar (score: {sem:.2f})"

    return 0.0, "no match found"

# ─────────────────────────────────────────
# 7. AGGREGATE FIT SCORE
# ─────────────────────────────────────────

def compute_fit_score(candidate_skills, required_skills):
    print("\nBuilding dynamic skill graph via Groq LLM...")
    graph = build_dynamic_graph(candidate_skills, required_skills)

    breakdown = []
    total = 0.0

    for req in required_skills:
        best_score = 0.0
        best_reason = "no match found"
        best_match = None

        for cand in candidate_skills:
            score, reason = skill_similarity_score(cand, req, graph)
            if score > best_score:
                best_score = score
                best_reason = reason
                best_match = cand

        total += best_score
        breakdown.append({
            "required_skill": req,
            "matched_with": best_match,
            "score": round(best_score * 100, 1),
            "reason": best_reason,
            "is_gap": best_score < 0.5
        })

    overall = round((total / len(required_skills)) * 100, 1)
    gaps = [item["required_skill"] for item in breakdown if item["is_gap"]]

    return overall, breakdown, gaps

# ─────────────────────────────────────────
# 8. DISPLAY
# ─────────────────────────────────────────

def display_results(overall_score, breakdown, gaps):
    print(f"\n{'='*70}")
    print(f"  OVERALL FIT SCORE: {overall_score}/100")
    print(f"{'='*70}")
    print(f"{'Required Skill':<22} {'Matched With':<22} {'Score':<8} Reason")
    print(f"{'-'*70}")

    for item in breakdown:
        gap_tag = " ← GAP" if item["is_gap"] else ""
        matched = item["matched_with"] or "—"
        print(f"{item['required_skill']:<22} {matched:<22} {item['score']:<8} {item['reason']}{gap_tag}")

    print(f"\nSkill Gaps ({len(gaps)}): {', '.join(gaps) if gaps else 'None'}")
    print(f"{'='*70}\n")

# ─────────────────────────────────────────
# 9. RUN
# ─────────────────────────────────────────

if __name__ == "__main__":
    candidate_skills = ["Flask", "PostgreSQL", "React", "Docker"]
    required_skills  = ["Python", "SQL", "JavaScript", "Kubernetes", "Django"]

    overall, breakdown, gaps = compute_fit_score(candidate_skills, required_skills)
    display_results(overall, breakdown, gaps)

Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 9123.06it/s]



Building dynamic skill graph via Groq LLM...
  [LLM] Querying implications for: 'Flask'
  [LLM] Querying implications for: 'PostgreSQL'
  [LLM] Querying implications for: 'React'
  [LLM] Querying implications for: 'Docker'

  OVERALL FIT SCORE: 57.6/100
Required Skill         Matched With           Score    Reason
----------------------------------------------------------------------
Python                 Flask                  95.0     Flask is a Python web framework, requiring solid Python knowledge
SQL                    PostgreSQL             96.0     PostgreSQL is an SQL-based relational database, requiring knowledge of SQL.
JavaScript             React                  97.0     React is a JavaScript library, requiring solid JavaScript knowledge
Kubernetes             —                      0.0      no match found ← GAP
Django                 —                      0.0      no match found ← GAP

Skill Gaps (2): Kubernetes, Django



In [11]:
pip install groq sentence-transformers scikit-learn networkx numpy


  Using cached groq-1.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached groq-1.2.0-py3-none-any.whl (142 kB)
Using cached pydantic-2.13.3-py3-none-any.whl (471 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 4.3 MB/s  0:00:00 eta 0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [groq]5/7 [pydantic]
Note: you may need to restart the kernel to use updated packages.


In [13]:
def check_two_skills(skill_a, skill_b):
    """Check similarity between any two skills directly."""
    print(f"\n  [LLM] Checking if '{skill_a}' implies '{skill_b}'...")

    prompt = f"""
You are a technical skill expert.

Does knowing "{skill_a}" imply or demonstrate knowledge of "{skill_b}"?
Also check the reverse: does knowing "{skill_b}" imply knowledge of "{skill_a}"?

Respond ONLY with a valid JSON object. No explanation, no markdown, no extra text.
Format:
{{
  "forward": {{
    "implies": true or false,
    "confidence": 0.0 to 1.0,
    "reason": "brief reason"
  }},
  "reverse": {{
    "implies": true or false,
    "confidence": 0.0 to 1.0,
    "reason": "brief reason"
  }}
}}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=500
    )

    text = response.choices[0].message.content.strip()
    text = re.sub(r"```json|```", "", text).strip()

    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        print(f"  [WARN] Could not parse response: {text}")
        result = None

    sem = semantic_similarity(normalize_skill(skill_a), normalize_skill(skill_b))

    print(f"\n{'='*55}")
    print(f"  SKILL SIMILARITY CHECK")
    print(f"{'='*55}")
    print(f"  Skill A : {skill_a}")
    print(f"  Skill B : {skill_b}")
    print(f"{'-'*55}")
    print(f"  Semantic Similarity  : {round(sem * 100, 1)}%")

    if result:
        fwd = result["forward"]
        rev = result["reverse"]

        fwd_score = round(fwd["confidence"] * 100, 1) if fwd["implies"] else 0.0
        rev_score = round(rev["confidence"] * 100, 1) if rev["implies"] else 0.0

        print(f"  '{skill_a}' → '{skill_b}' : {fwd_score}%  ({fwd['reason']})")
        print(f"  '{skill_b}' → '{skill_a}' : {rev_score}%  ({rev['reason']})")

        # Overall similarity = best of all three signals
        best = max(sem * 100, fwd_score, rev_score)
        print(f"{'-'*55}")
        print(f"  Overall Similarity   : {round(best, 1)}%")

    print(f"{'='*55}\n")

In [15]:
skill_a=input("skill a:")
skill_b=input("skill b:")
print(check_two_skills(skill_a,skill_b))

skill a: python
skill b: java



  [LLM] Checking if 'python' implies 'java'...

  SKILL SIMILARITY CHECK
  Skill A : python
  Skill B : java
-------------------------------------------------------
  Semantic Similarity  : 45.0%
  'python' → 'java' : 0.0%  (different syntax and paradigms)
  'java' → 'python' : 0.0%  (different syntax and paradigms)
-------------------------------------------------------
  Overall Similarity   : 45.0%

None
